In [0]:
%run ../00_common/01_environment_config

In [0]:
%run ../00_common/02_bronze_helpers

In [0]:
silver_table = f"{catalog_name}.{silver_schema}.sprints"
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
sprints_table_read = spark.table(bronze_table)
                    

In [0]:
from pyspark.sql import functions as f
sprints_table_final = (sprints_table_read
                      .drop("url")
                      .withColumnsRenamed({
                          "date":"race_date",
                          "raceName":"race_name",
                            "constructorId":"constructor_id",
                            "driverId":"driver_id",
                            "grid":"grid_position",
                            "laps":"completed_laps",
                            "number":"car_number",
                            "position":"finish_position",
                            "positionText":"finish_position_text"
                      })
                      .filter(
                          (f.col("season").isNotNull()) &
                          (f.col("round").isNotNull()) &
                          (f.col("constructor_id").isNotNull())&
                          (f.col("driver_id").isNotNull())
                      )
                      .dropDuplicates(["season","round","constructor_id", "driver_id"])
                      .withColumn("race_name",f.initcap(f.col("race_name")))
)
                      

In [0]:
display(sprints_table_read.count()- sprints_table_final.count())

In [0]:
(sprints_table_final.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))